# Week 07 — Home exercise 1: Predict the output

**Solution proposal.**

For each snippet: what it displays, and why. Then the two broken snippets.

In [1]:
import numpy as np
import pandas as pd

## a) Array types

Displays `[1. 2. 3.5]`, then `['1' '2' '3']`, then `float64`.

An array holds **one** type. When the values you hand it do not agree, NumPy does not keep the
mixture — it picks a type that can hold all of them.

Whole numbers can be written as decimals without losing anything, so the first becomes `float64`.
Nothing can hold a number and a string except text, so the second becomes text — note the quotes
around the `1` and the `2`. They are no longer numbers and no longer add up.

The third is the one that matters later. Those values are whole numbers, but `np.nan` is a
**decimal**, so the array has to be `float64` to hold it. This is why a column of counts with one
value missing comes back as a float column.

In [2]:
print(np.array([1, 2, 3.5]))
print(np.array([1, 2, "3"]))
print(np.array([12, 14, np.nan]).dtype)

[1.  2.  3.5]
['1' '2' '3']
float64


## b) Alignment

pandas matched the two Series on their **labels**, not their positions.

| label | | |
|---|---|---|
| Denmark | 30 + 1 | 31 |
| Norway | 10 + 2 | 12 |
| Sweden | 20 + ? | `NaN` — only in `a` |
| Finland | ? + 3 | `NaN` — only in `b` |

Three things to notice. Position was irrelevant: Norway is first in one Series and second in the
other, and it still lined up correctly. Labels present on only one side produce `NaN` rather than an
error or a dropped row. And the result comes back in alphabetical order, which is neither input's
order, because pandas built a new index out of the union of both.

This is the explanation for most "why is my column all `NaN`?" questions.

In [3]:
a = pd.Series([10, 20, 30], index=["Norway", "Sweden", "Denmark"])
b = pd.Series([1, 2, 3], index=["Denmark", "Norway", "Finland"])

print(a + b)

Denmark    31.0
Finland     NaN
Norway     12.0
Sweden      NaN
dtype: float64


## c) `.loc` against `.iloc`

Displays `3`, then `2`.

`.loc` asks for the rows **labeled** 1 to 3, and there is no sensible way to include a label "up to
but not including" it — either you want row 3 or you do not. So `.loc` is inclusive at both ends:
rows 1, 2 and 3.

`.iloc` asks for **positions** 1 to 3, and positions in Python have always stopped before the end
value, exactly as `"Python"[0:4]` gives four characters and `range(5)` stops at 4. So `.iloc` gives
positions 1 and 2.

The two look confusable here only because the index happens to be numbers. One is a name that looks
like a number; the other is a count.

In [4]:
df = pd.DataFrame({"x": [10, 20, 30, 40, 50]})

print(len(df.loc[1:3]))
print(len(df.iloc[1:3]))

3
2


## d) Chained assignment

Displays the **original** table, with Oslo still at 4.2 — plus a `ChainedAssignmentError` warning.

`df[df["city"] == "Oslo"]` builds a **new** table containing the Oslo row. The assignment then
changes that new table, which nothing refers to, so it is thrown away immediately. The original
never saw it.

This is chained assignment: two square-bracket operations in a row with an `=` at the end. It never
works, and pandas says so rather than letting you believe it did.

In [5]:
df = pd.DataFrame({"city": ["Oslo", "Bergen"], "temp": [4.2, 7.8]})

df[df["city"] == "Oslo"]["temp"] = 99

print(df)

     city  temp
0    Oslo   4.2
1  Bergen   7.8


C:\Users\s14754\AppData\Local\Temp\ipykernel_23472\748128175.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  df[df["city"] == "Oslo"]["temp"] = 99


The fix is to select rows and column in **one** step with `.loc`.

In [6]:
df.loc[df["city"] == "Oslo", "temp"] = 99

print(df)

     city  temp
0    Oslo  99.0
1  Bergen   7.8


## e) `and` between two masks

Raises `ValueError: The truth value of a Series is ambiguous.`

`and` has to decide whether its left-hand side is true or false, as a single answer, so that it knows
whether to bother evaluating the right-hand side. `df["year"] == 2020` is not one answer — it is one
answer per row — and pandas refuses to guess which one you meant.

`&` is the version that works element by element, and it needs brackets around each condition
because `&` binds more tightly than `==` does.

In [7]:
df = pd.DataFrame({"year": [2020, 2021], "value": [5, 15]})

try:
    print(df[df["year"] == 2020 and df["value"] > 10])
except ValueError as error:
    print(f"ValueError: {error}")

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().


Written correctly it runs — and returns nothing, because no row satisfies both conditions.
An empty result is an answer, not an error.

In [8]:
print(df[(df["year"] == 2020) & (df["value"] > 10)])

Empty DataFrame
Columns: [year, value]
Index: []


## f) `astype(int)` on a column with holes

Displays `float64`, then raises `IntCastingNaNError`.

`Age` looks like whole numbers and is stored as `float64`, because 177 of the 891 ages are missing,
missing is `NaN`, and `NaN` is a float. Same rule as (a).

`astype(int)` then fails, and it is right to. There is no integer that means "missing", so converting
would have to invent a value — 0, or -1, or the average — and pandas will not pick one for you. The
error is asking you to make a decision.

In [9]:
titanic = pd.read_csv("../data/titanic.csv")

print(titanic["Age"].dtype)
print("missing ages:", titanic["Age"].isna().sum())

try:
    titanic["Age"].astype(int)
except Exception as error:
    print(f"{type(error).__name__}: {str(error)[:70]}")

float64
missing ages: 177
IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer.Replace or rem


Two defensible repairs. They give different answers, which is the point.

In [10]:
print("drop:", len(titanic.dropna(subset=["Age"])), "rows kept")
print("fill:", titanic["Age"].fillna(titanic["Age"].median()).astype(int).head(3).tolist())

drop: 714 rows kept
fill: [22, 38, 26]


## Broken snippet 1: the brackets are missing

`&` binds more tightly than `>` and `==`, so Python reads

```python
co2["year"] == (2023 & co2["co2_total"]) > 100
```

which is not what anybody meant, and raises a `TypeError` about combining a float array with a bool.
The operator precedence is the same rule we met when building conditions in Part 1 — it has simply
moved to a new place.

**Every condition gets its own brackets. Always.**

In [11]:
co2 = pd.read_csv("../data/co2_emissions.csv")

try:
    co2[co2["year"] == 2023 & co2["co2_total"] > 100]
except TypeError as error:
    print(f"TypeError: {str(error)[:80]}")

subset = co2[(co2["year"] == 2023) & (co2["co2_total"] > 100)]

print("rows:", len(subset))

TypeError: Cannot perform 'rand_' with a dtyped [float64] array and scalar of type [bool]
rows: 78


## Broken snippet 2: it runs, and answers a different question

Nothing filters to 2023. `co2["co2_total"].mean()` averages every country in every year from 2000 to
2023, and the f-string then labels that number as the 2023 average. No error, no warning, and a
report that is simply wrong.

This is the failure mode worth being afraid of. An exception tells you where to look; a wrong number
does not.

In [12]:
wrong = co2["co2_total"].mean()
right = co2[co2["year"] == 2023]["co2_total"].mean()

print(f"all years: {wrong:.1f}")
print(f"2023 only: {right:.1f}")

all years: 1063.6
2023 only: 1301.4


## The pattern behind (b), (d) and broken snippet 2

None of those three raised an error, and all three gave a wrong answer.

- **(b)** produced `NaN` where you expected numbers, because the labels did not match and pandas told
  you so in the only way it can.
- **(d)** produced no change at all, and would have been silent about it if pandas had not added a
  warning specifically for this case.
- **(2)** produced a perfectly reasonable-looking number for the wrong rows.

The habit that catches all three: after any step that filters, joins or combines, check the **shape**
and check a value you already know. `len(df)` before and after a filter costs nothing and is the
single most useful check in pandas. A wrong number that looks right will otherwise travel all the way
into your conclusions.